# 00 · Setup

**Purpose**
- Connect Colab to Google Drive
- Get the GitHub repo into Drive
- Install packages and check the GPU
- Save this notebook back to GitHub

**How to use**
- Runtime → Change runtime type → **L4** (or A100)
- Runtime → **Run all**

## 1 · Mount Google Drive
- Code and data live in Drive, so nothing is lost when the session ends.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2 · Get the repo
- Clones the repo into Drive on the first run; skips if already there.
- Token comes from Colab secrets (🔑 → `GITHUB_TOKEN`) and is never saved to Drive.
- Commit email is set to your private GitHub noreply address automatically.

In [2]:
import os, json, subprocess, urllib.request
from google.colab import userdata

GITHUB_USER = "dedeepyakm"
REPO        = "cw-mpvr"
REPO_DIR    = f"/content/drive/MyDrive/{REPO}"
token       = userdata.get("GITHUB_TOKEN")

if not os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "clone", f"https://{token}@github.com/{GITHUB_USER}/{REPO}.git", REPO_DIR], check=True)
    subprocess.run(["git", "-C", REPO_DIR, "remote", "set-url", "origin", f"https://github.com/{GITHUB_USER}/{REPO}.git"], check=True)
    print("Cloned into", REPO_DIR)
else:
    print("Repo already in Drive:", REPO_DIR)

# noreply email keeps your real email private
try:
    req = urllib.request.Request("https://api.github.com/user", headers={"Authorization": f"Bearer {token}"})
    me = json.load(urllib.request.urlopen(req))
    email = f"{me['id']}+{me['login']}@users.noreply.github.com"
except Exception:
    email = f"{GITHUB_USER}@users.noreply.github.com"

%cd {REPO_DIR}
!git config user.name "Dedeepya Korukonda"
!git config user.email "{email}"
print("Commit email:", email)

Repo already in Drive: /content/drive/MyDrive/cw-mpvr
/content/drive/MyDrive/cw-mpvr
Commit email: 268308496+dedeepyakm@users.noreply.github.com


## 3 · Install packages
- PyTorch is preinstalled on Colab; this adds the rest.

In [3]:
!pip install -q -r requirements.txt

## 4 · Standard setup cell
- Same cell goes at the top of every notebook (after mounting Drive).
- Makes `src/` importable, loads the config, fixes the seed, shows the GPU.
- No `autoreload`: it breaks on Colab's Python 3.13 (ERROR_LOG E01). After editing `src/`, restart the session.

In [4]:
import sys
REPO_DIR = "/content/drive/MyDrive/cw-mpvr"
%cd {REPO_DIR}
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from src.utils import load_config, set_seed, gpu_info
from src.errorlog import log_error
from src.gitsync import save_and_push

cfg = load_config()
set_seed(cfg["seed"])
print(gpu_info())

/content/drive/MyDrive/cw-mpvr
NVIDIA L4 | 23.7 GB | torch 2.11.0+cu130


## 5 · Check the folders
- `cw-mpvr/` → code, in git
- `cw-mpvr-data/` → GQA files, images, cached outputs, **not** in git

In [5]:
for k, v in cfg["paths"].items():
    print(f"{k:6s} -> {v}  (exists: {os.path.isdir(v)})")

repo   -> /content/drive/MyDrive/cw-mpvr  (exists: True)
data   -> /content/drive/MyDrive/cw-mpvr-data  (exists: True)
cache  -> /content/drive/MyDrive/cw-mpvr-data/cache  (exists: True)


## 6 · Tidy the repo
- Removes a file saved by mistake earlier.
- Removes duplicate entries from `ERROR_LOG.md`.
- Safe to rerun: does nothing if already tidy.

In [6]:
import re

stray = "notebooks/01_data_eda.ipynb"
if os.path.exists(stray):
    subprocess.run(["git", "rm", "-q", stray], check=True)
    print("Removed", stray)

log = open("ERROR_LOG.md").read()
head, *entries = re.split(r"(?=\n### E\d+)", log)
seen, keep = set(), []
for e in entries:
    m = re.search(r"\*\*Error\*\*: (.*)", e)
    key = m.group(1) if m else e
    if key not in seen:
        seen.add(key); keep.append(e)
open("ERROR_LOG.md", "w").write(head + "".join(keep))
print(f"ERROR_LOG.md: {len(keep)} entr{'y' if len(keep)==1 else 'ies'}")

Removed notebooks/01_data_eda.ipynb
ERROR_LOG.md: 1 entry


## 7 · Logging errors (reference)
- When a cell fails: fix it, then call `log_error(...)` once in a new cell, then delete that cell.
- Everything collects in `ERROR_LOG.md` for the report's "Errors and fixes" section.

In [7]:
# log_error(
#     where="notebook / step",
#     cause="why it happened",
#     fix="what solved it",
# )

## 8 · Save to GitHub
- Saves this notebook into the repo, then commits and pushes everything.
- Last cell of every notebook.

In [ ]:
save_and_push("notebooks/00_setup.ipynb", "Clean setup notebook")